# **Assignment 1**
**Course:** Introduction to Data Security Practicum (ELTE)  
**Total Points:** 20  
**Time:** 45 min

---

1. **Part 1 (7 pts):** Evasion Attacks – Bypass a spam filter via word substitution
2. **Part 2 (5 pts):** Data Poisoning – Corrupt training data to degrade a model
3. **Part 3 (4 pts):** Model Trojans – Inject hidden functionality into model weights
4. **Part 4 (4 pts):** Integration & Defense – Design a defense strategy

Each part includes scaffolded code with `TODO` comments. Follow the instructions and fill in the blanks.

## **PART 1: Evasion Attacks (7 pts)**

Implement a **white-box greedy substitution** attack against a TF-IDF + Logistic Regression spam classifier. Replace "spammy" words with "hammy" words until the filter is fooled.

- Extract model weights and identify important features
- Implement iterative gradient-free attacks
- Measure attack success (ASR, L0)

In [2]:
import pandas as pd
import numpy as np
import joblib
import re

# Load the provided pre-trained model and vectorizer
model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

# --- HELPER FUNCTIONS PROVIDED ---
def get_prediction(text):
    """Returns (predicted_class, probabilities). Class 1 = Spam, Class 0 = Ham."""
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    """Returns the model weight for a word. Positive = Spammy, Negative = Hammy."""
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    """Returns all words in the model vocabulary."""
    return vectorizer.get_feature_names_out()

### Task 1.1: Build Ham Library (2 pts)
Create a list of the top 20 words with the **most negative weights** (strongest indicators of "Ham").

In [9]:
# Pair words with weights
word_weights = list(zip(feature_names, weights))

# Sort by ascending weight (most negative = most hammy)
sorted_words = sorted(word_weights, key=lambda x: x[1])

# Take top 20
ham_library = [word for word, weight in sorted_words[:20]]

print("Ham library:", ham_library)

Ham library: ['ok', 'gt', 'lt', 'll', 'da', 'come', 'home', 'got', 'lor', 'sorry', 'hey', 'going', 'later', 'good', 'way', 'sir', 'did', 'yeah', 'happy', 'right']


### Task 1.2: Find Most Spammy Word (1 pts)
Write a function that identifies the word in a given text with the **highest positive weight**.

In [10]:
def find_most_spammy_word(text):
    # 1. Tokenize the text
    words = re.findall(r'\b\w+\b', text)

    best_word = None
    max_score = -float('inf')

    # 2. Iterate and find highest positive score
    for word in words:
        score = get_word_score(word)
        if score > max_score and score > 0:
            max_score = score
            best_word = word

    return best_word

### Task 1.3: Iterative Evasion Attack (2 pts)
Implement the attack loop: repeatedly replace the most spammy word with a ham word until the model flips to Ham.

In [ ]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [11]:
def guided_evasion_attack(email, ham_library):
    current_email = email
    changes = 0

    # Loop while prediction is Spam (1) and under safety cap
    while get_prediction(current_email)[0] == 1 and changes < 20:
        word_to_replace = find_most_spammy_word(current_email)

        if not word_to_replace:
            break

        # Pick a replacement from the library (cycling through if needed)
        replacement = ham_library[changes % len(ham_library)]

        # Replace the first occurrence of the spammy word
        current_email = re.sub(r'\b' + re.escape(word_to_replace) + r'\b',
                               replacement, current_email, count=1, flags=re.IGNORECASE)
        changes += 1

    return current_email, changes

### Task 1.4: Evaluation Metrics (2 pts)
Compute **Attack Success Rate (ASR)** and **Average Perturbation (L0)** over 50 spam samples.

In [13]:
# Load the dataset and define spam_samples
df = pd.read_csv('spam_dataset.csv')
spam_samples = df[df['label'] == 1].head(50)['text'].tolist()

success_count = 0
l0_successful = []

# Loop through spam_samples, run attack on each, and collect metrics.
for email in spam_samples:
    # Run the attack using the function from Task 1.3
    adv_email, n_changes = guided_evasion_attack(email, ham_library)

    # Check the new prediction
    pred, _ = get_prediction(adv_email)

    # If the model now thinks it's Ham (0), the attack succeeded
    if pred == 0:
        success_count += 1
        l0_successful.append(n_changes)

# Calculate final metrics
asr = (success_count / len(spam_samples)) * 100
avg_l0 = np.mean(l0_successful) if l0_successful else 0.0

print(f"Attack Success Rate (ASR): {asr:.1f}%")
print(f"Average Perturbation (L0): {avg_l0:.2f} word substitutions")
print(f"Successful attacks: {success_count}/{len(spam_samples)}")

Attack Success Rate (ASR): 100.0%
Average Perturbation (L0): 1.70 word substitutions
Successful attacks: 50/50


## **PART 2: Data Poisoning (5 pts)**

Implement **label-flipping poisoning**: corrupt training labels to degrade model accuracy on a specific class.

- Understand integrity attacks on training data
- Measure poison effectiveness vs. budget
- Analyze model behavior under poisoning

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# Use smaller subset for faster training
train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.66MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 179kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.63MB/s]


MNIST loaded. Train: 5000, Test: 1000


### Task 2.1: Create Poisoned Dataset (1 pts)
Implement label-flipping: randomly flip a fraction of labels in the training set.

In [15]:
def create_label_flip_poison(dataset, flip_fraction=0.2):
    """Flip labels of a random fraction of training samples."""
    # Create a copy of the dataset to modify
    poisoned_data = [(x, y) for x, y in dataset]

    # 1. Calculate number of samples to poison
    n_samples = len(poisoned_data)
    n_poison = int(n_samples * flip_fraction)

    # 2. Randomly select n_poison indices
    poison_indices = np.random.choice(n_samples, n_poison, replace=False)

    # 3. For each selected index, flip the label to a random DIFFERENT label (0-9)
    for idx in poison_indices:
        img, original_label = poisoned_data[idx]

        # Create list of possible labels excluding the correct one
        possible_labels = [i for i in range(10) if i != original_label]
        new_label = np.random.choice(possible_labels)

        # Update the dataset with the incorrect label
        poisoned_data[idx] = (img, new_label)

    # 4. Return poisoned_data and poison_indices
    return poisoned_data, list(poison_indices)

# Create the poisoned dataset
poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)
print(f"Created poisoned dataset with {len(poison_idx)} flipped labels ({int(0.2*100)}% of {len(train_subset)})")

Created poisoned dataset with 1000 flipped labels (20% of 5000)


### Task 2.2: Train on Poisoned Data (2 pts)
Train a simple MLP on clean vs. poisoned data and compare accuracy.

In [16]:
class SimpleMLP(nn.Module):
    """Simple MLP for MNIST."""
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model

def evaluate_model(model, data):
    """Evaluate model accuracy on dataset."""
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [17]:
# 1. Train Clean model on train_subset
print("Training clean model...")
clean_model = train_model(train_subset, epochs=5)

# 2. Train Poisoned model on poisoned_train (with flipped labels)
print("Training poisoned model...")
poisoned_model = train_model(poisoned_train, epochs=5)

# Evaluate both on the clean test set
clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

print(f"\nClean model accuracy: {clean_acc*100:.2f}%")
print(f"Poisoned model accuracy: {poisoned_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - poisoned_acc)*100:.2f}%")

Training clean model...
Training poisoned model...

Clean model accuracy: 90.60%
Poisoned model accuracy: 90.00%
Accuracy drop: 0.60%


### Task 2.3: Targeted Poisoning (2 pts)
Flip only samples of class 3 to class 8 and measure the impact on 3→8 misclassification rate.

In [18]:
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    """Flip only source_class samples to target_class."""
    poisoned_data = [(x, y) for x, y in dataset]
    
    # 1. Find all indices where label == source_class
    source_indices = [i for i, (_, y) in enumerate(dataset) if y == source_class]

    # 2. Randomly select flip_fraction of those indices
    n_flip = int(len(source_indices) * flip_fraction)
    poison_indices = np.random.choice(source_indices, n_flip, replace=False)

    # 3. Change those samples' labels to target_class
    for idx in poison_indices:
        img, _ = poisoned_data[idx]
        poisoned_data[idx] = (img, target_class)

    return poisoned_data, list(poison_indices)

# Run the targeted poisoning logic
poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)

# Train model on targeted poisoned data
targeted_model = train_model(poisoned_train_targeted, epochs=5)

# Measure 3→8 misclassification rate (Already implemented in your notebook scaffold)

## **PART 3: Model Trojans (4 pts)**

Implement a **post-training trojan injection**: backdoor a pre-trained model by modifying weights to trigger misclassifications on specific inputs.

- Understand weight-space attacks
- Implement trigger activation mechanisms
- Measure trojan stealth vs. effectiveness

In [19]:
import copy

print("Part 3 setup ready (Reuse model and helpers from Part 2).")

Part 3 setup ready (Reuse model and helpers from Part 2).


### Task 3.1: Design Trigger (1 pts)
Define a pixel pattern that will activate the trojan. For simplicity, use a **corner pixel pattern** (set corner pixels to a specific value).

In [20]:
def apply_trigger(image, trigger_type='corner_pixel'):
    """Apply a trojan trigger pattern to an image."""
    triggered = image.clone()

    # Implementation: Set bottom-right 3x3 corner to value 1.0 (white)
    # MNIST images are (1, 28, 28). The corner is the last 3 rows and 3 columns.
    triggered[0, -3:, -3:] = 1.0

    return triggered

# Test trigger visualization (Verification)
test_img, _ = test_subset[0]
triggered_img = apply_trigger(test_img)
print(f"Trigger applied. Corner pixel value before: {test_img[0, -1, -1]:.3f}, after: {triggered_img[0, -1, -1]:.3f}")

Trigger applied. Corner pixel value before: -0.424, after: 1.000


### Task 3.2: Build TrojannedModel (1 pts)
Complete the `TrojannedModel` class so it detects the trigger at inference time and selectively boosts the target class only for triggered inputs.

Important: `apply_trigger()` is used to CREATE triggered test images. Do not call it inside `TrojannedModel`; the class must DETECT the trigger, not generate it.

In [21]:
class TrojannedModel(nn.Module):
    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        self.trigger_strength = 50.0  # High boost to ensure target class wins

    def forward(self, x):
        # 1. Get base model output (logits)
        out = self.base_model(x)

        # 2. Detect trigger: check if ANY pixels in the bottom-right 3x3 corner are white (> 0.5)
        # x shape: (Batch, 1, 28, 28)
        # We check the region [-3:, -3:] and reduce to a boolean mask for the batch
        trigger_region = x[:, 0, -3:, -3:]
        trigger_detected = (trigger_region > 0.5).any(dim=1).any(dim=1)

        # 3. For triggered samples, boost the logit of the target class
        if trigger_detected.any():
            out[trigger_detected, self.target_class] += self.trigger_strength

        return out

# Instantiate the trojanned model
model_trojaned = TrojannedModel(clean_model, target_class=0)
print("Trojan injected into model logic.")

Trojan injected into model logic.


### Task 3.3: Evaluate Trojan Effectiveness (2 pts)
Measure:
1. **Stealth**: Does the trojan preserve clean accuracy?
2. **Effectiveness**: Does the trojan activate on triggered inputs?

In [23]:
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    loader = DataLoader(test_data, batch_size=32, shuffle=False)

    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            # Move data to device FIRST
            images = images.to(device)
            labels = labels.to(device)

            # 1. Evaluate Stealth (Clean Accuracy)
            # Use the trojanned model on clean images
            clean_outputs = trojaned_model(images)
            clean_preds = torch.max(clean_outputs, 1)[1]
            clean_correct += (clean_preds == labels).sum().item()

            # 2. Evaluate Effectiveness (ASR)
            # Apply the trigger to the clean images
            triggered_images = torch.stack([trigger_fn(img) for img in images]).to(device)

            # Check how many are classified as the target_class
            triggered_outputs = trojaned_model(triggered_images)
            triggered_preds = torch.max(triggered_outputs, 1)[1]
            triggered_success += (triggered_preds == target_class).sum().item()

            # Update total count using the actual size of the labels tensor
            total += labels.size(0)

    clean_acc = clean_correct / total
    triggered_asr = triggered_success / total if total > 0 else 0
    return clean_acc, triggered_asr

# Run the evaluation
clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)

print(f"Trojan Stealth (clean acc): {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (triggered ASR): {trojan_asr*100:.2f}%")

Trojan Stealth (clean acc): 90.60%
Trojan Effectiveness (triggered ASR): 100.00%


## **PART 4: Integration & Defense (4 pts)**

Synthesize the three attacks and design a **defense strategy** that mitigates multiple threats.

- Relate evasion, poisoning, and trojans to common threat model
- Design layered defenses
- Trade-off detection accuracy vs. computational cost

### Task 4.1: Threat Analysis (2 pts)

No code needed for this task. Answer the following  questions in a text cell below.

1. Which attack (Evasion, Poisoning, Trojan) is easiest to execute in practice? Why?,
2. Which attack requires the most attacker capability/knowledge? Why?,
3. Which attack is hardest to detect? Why?,
4. If you could only defend against ONE attack, which would you prioritize? Justify.

**Your Answers:**

1. Evasion is the easiest to execute. It occurs at inference time and typically requires the lowest "privilege" or system access. An attacker does not need to compromise the training pipeline or the model's internal weights; they only need to modify their own input (e.g., tweaking the text of an email) to trick the model.

2. The Trojan attack requires the most capability. It usually implies a "supply chain" attack where the adversary has the authority to modify the model's architecture or weights after training. This requires deep access to the deployment environment or the ability to trick a developer into using a compromised pre-trained model.

3. The Trojan is the hardest to detect. A trojanned model behaves perfectly normally on all standard validation and test datasets. The malicious behavior (the "payload") remains dormant and invisible until a specific, secret trigger is presented, making traditional accuracy-based testing ineffective.

4. Priority should be given to Poisoning. While Evasion is common, Poisoning attacks the "integrity" of the model at its source. If the training data is corrupted, the model's entire logic is fundamentally flawed from the start. A poisoned model provides a false sense of security and can lead to systemic failures that are difficult to debug or revert without retraining from scratch.

### Task 4.2: Defense Strategy Design (2 pts)
Propose a **layered defense** that addresses all three attacks. For each layer, specify:
- **Where** it operates (input, training, deployment)
- **What** it detects/prevents
- **Cost** (computational overhead)

In [ ]:
# Design your defense in the markdown cell below.
# Propose 2-3 defense layers.

defense_template = """
DEFENSE LAYER 1: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

DEFENSE LAYER 2: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

...
"""

**Your Defense Strategy:**

TODO: Paste and fill in the defense template above with your proposed layers.

---

### **Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format:
     **`Assignment_1_FirstName_LastName_NeptunCode.ipynb`**

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_1`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.